In [ ]:
"""
Do we want 1 or 4 Vpp for new sensors?

4 Vpp allows a higher dynamic range of sensor, reducing issues of saturation.
However, bin width increases by a factor of four.

What is the probability of saturation events making a TGF measurement mostly useless versus the
impact of lower bit precision on spectrum estimation. Saturation events are less impactful the more
events we capture because the marginal value of "good" measurements will decrease with increasing data.
How much "more" of the spectrum would 4Vpp let us capture? Ofc, pileup also causes increased voltage...

On the other hand... due to power law spectrum, higher energy photons have higher marginal value for estimating spectrum...
Is it better to capture lower energy spectrum with more accuracy or higher energy spectrum

P(saturation) <- spectrum (Multiple from different event types?),
              <- count rate <- distance (we dont know altitudes they occur which is the toughest part)

Consider from signals perspective! Perhaps we can estimate spectrum by looking at histogram of derivatives and instantaneous magnitude?

Consider how to do regularization
- L1 nice in theory, but still limited
- Penalizing adjacent counts would be better

Try convolving just the front of response with trace rather than the whole response. Might tell us more about the magnitude
Also maybe we just need to sum adjacent deconv or assume that peaks only include singe events
Holland suggests deep learning lol (look up Michigan deep learning pulse pileup)
    
"""


In [ ]:
"""
Need to investigate pileup and saturation? Using sim data (ignoring the THOR issue)
How much faster does pileup degrade error? Relative impact vs saturation?

Plan for this analysis:

for countrate in countrates:
    generate trace
    NNLSR(trace)
    
    metrics
        error per bin?
        % photons counted?
        % energy counted?
        
Analysis:
    if metric per countrate is scalar: plot(countrate, metric)
    if metric per countrate is vector: loop : plot(countrate, metric vector)
    
    
Repeat with Short response (just ignore 1V ceiling and we can look at improvements to pileup.. behavior...

"""

In [ ]:
import numpy as np
import numpy.random as ran
import matplotlib.pyplot as plt
import pylab as pl
import pandas as pd
import scipy
from scipy.interpolate import CubicSpline

import time
from matplotlib import rcParams
from util.Processing import *
from util.DataGen import *

In [ ]:
#variables for creating the trace
fwhm = 10 #50.0 
TGF_duration = fwhm*3e-6
# counts = int(countrate*TGF_duration) #total counts incident on the detector   
#fwhm = 50.0 #fwhm of the total TGF count distribution in units microseconds
#TGF_duration = fwhm*3e-6 #this is a really rough estimate to get a rough estimate of the count rate
#countrate = int(counts/TGF_duration)
mean = .7 #mean of the TGF trace distribution
std = .5 #detemines the amount of asymetry in the TGF trace distribution
dt = 1e-9 # seconds before pulse
tstep = 25e-9 #sampling rate in seconds. 40MHz
trace_length = 28000 #number of samples in a trace file (700us at 40MHz)
keV_per_area = .147 #determined by trial and error to match energy range of instrument 
mV_per_ADC = 1000./4096.
specscale_keV = 5.0  #spectrum scaling i.e. keV/line in the spectrum file
baseline = 0 #110
basenoise = 1 #units mV
bits = 14  #use 12 for doing listmode but use 10 to compare traces to real trace files

#variables for integrating trace pulses into listmode events
thresh = 8.0     #units of mV  this is the pulse trigger threshold
int_i = 50      #integ.ration time = 1.25 microsecs = 50 samples at 40MHz sampling 
dead_i = int_i     #deadtime = integration time
extend = 1    #extendable dead time parameter
escale = .63  #being used to scale the pulse integration value to energy in keV. experimentally determined.

#example spectrum of TGF energy deposit in a detector
NaI_Response = np.loadtxt('../original/NaI_Response',usecols=(1),dtype=float)
bins = np.loadtxt('../original/NaI_Response',usecols=(0),dtype=float)
#s = np.genfromtxt('/home/enp//Desktop/Emorpho Analysis Software and Calibration data/Emorpho Simulations/alt5SFT_noaa_plane_rough.out', usecols = (2), skip_footer=2)

spectrum = NaI_Response
binenergies = bins*1e3 #units keV 
#spectrum[49]= 2000 #49 is 1000keV, 85 is 6800keV

#real NaI trace example data
tracedata = pd.read_csv('../original/real_nai_trace.csv')

In [ ]:
#calling the sample trace pulse and real trace data
pulsetimes, pulse = nai_pulse(1.)
nsamples = len(pulse)
tdatatime = tracedata.Seconds[253:350]-tracedata.Seconds[253]
tdata = (tracedata.Tracesample[253:350]-27)/(max(tracedata.Tracesample)-27)

# fit single trace plot 
plt.figure(figsize=(8,5), dpi=200)
plt.plot(tdatatime*1e6,tdata,color='red',label='Real trace pulse')
plt.plot(pulsetimes*1e6,pulse,color='black',label='Modeled trace pulse')

def kernel_stretch(kernel, time, stretch_factor, spline_):
    compressed_time = np.linspace(0, pulsetimes[-1], int(time.size*stretch_factor))
    compressed_pulse = spline_(compressed_time)
    compressed_time *= stretch_factor
    return compressed_time, compressed_pulse

spline = CubicSpline(x=pulsetimes, y=pulse)

factor = 1/2
compressed_time, compressed_pulse = kernel_stretch(pulse, pulsetimes, factor, spline)
plt.plot(compressed_time*1e6, compressed_pulse, color='blue',label='{}x stretched trace pulse'.format(factor), alpha=.7)

factor = 1/4
compressed_time, compressed_pulse = kernel_stretch(pulse, pulsetimes, factor, spline)
plt.plot(compressed_time*1e6, compressed_pulse, color='green',label='{}x stretched trace pulse'.format(factor), alpha=.7)

factor = 4
stretched_time, stretched_pulse = kernel_stretch(pulse, pulsetimes, factor, spline)
plt.plot(stretched_time*1e6, stretched_pulse, color='cyan',label='{}x stretched trace pulse'.format(factor), alpha=.7)

plt.xlabel('microseconds',fontsize=16)
plt.title('Modeled NaI pulse',fontsize=16)
#plt.xlim(-.5e-6,3.0e-6)
plt.tick_params(labelsize=12)
plt.legend(fontsize=12)

plt.xlim(0, 3)




In [ ]:
countrates = np.logspace(5,9,9)

vpps = [1, 4]
gain_adjust = 1
factor = 1
repeat = 1
error_threshold = 5

show_full = False
saturation = True

plot_trace = True
plot_true_volts = False
plot_deconv = False
plot_error = False
plot_hists = True

def my_histplot(axis_, hist, bin_edges, alpha, label):
    x = np.stack((bin_edges[1:], bin_edges[:-1])).transpose()[:,::-1].ravel() 
    y = np.stack((hist, hist)).transpose().ravel()
    x = np.insert(x, 0, 0)
    y = np.insert(y, 0, 0)
    axis_.plot(x, y, alpha=alpha, label=label)

for countrate in countrates:
    
    rows = int(plot_trace) + int(plot_true_volts) + int(plot_deconv) + int(plot_error) + int(plot_hists)
    fig, axes = plt.subplots(rows, len(vpps), figsize=(10,3*rows), dpi=200)
    fig.suptitle('Countrate {:.2E}'.format(countrate))
    
    for col, vpp in enumerate(vpps):
        bins = np.linspace(1, vpp*1000, 2**bits)
        true = np.array(bins.size)
        counted = np.array(bins.size)
        
        for n in np.arange(repeat):
        
            axis_count = 0

            counts = int(countrate*TGF_duration) #total counts incident on the detector

            time, kernel = kernel_stretch(pulse, pulsetimes, factor, spline)

            # note that mV_per_ADC, keV_per_area adjusted by Vpp to preserve pulse height to energy ratio, while decreasing fwhm of response
            _ = lognormal_nai_trace(counts=counts, fwhm=fwhm, spectrum=spectrum,binenergies=binenergies, dt=dt, tstep=tstep,
                                     trace_length = trace_length,
                                     mV_per_ADC=gain_adjust * vpp * factor * mV_per_ADC,
                                     keV_per_area=gain_adjust * vpp * keV_per_area,
                                     specscale_keV=specscale_keV, baseline=baseline, basenoise=basenoise,
                                     bits=bits, mean=mean,std=std, kernel=kernel, vpp=vpp,
                                     saturation=saturation, seed=0)
            trace_time, trace, photon_index, energies, peak_volts = _
            
            # axes[0].plot(trace)
            # assert False

            # Plot Window Selection #########################################################################################
            trace_time *= 1E6 # convert to microseconds    
            trigger_time, trigger_index = trace_trigger(trace, trace_time)
                
            if not show_full:
                # Get Trace slice with significant activity
                
                if fwhm == 10:
                    lower = 100
                    upper = 150
                else:
                    print('Need to define good window for this TGF FWHM')
                    assert False
                    
                mask = np.logical_and(lower <= trace_time, trace_time <= upper)
                trace_slice = trace[mask]
                trace_time_slice = trace_time[mask]
    
                lower = np.nonzero(lower <= trace_time)[0][0] #first
                upper = np.nonzero(trace_time <= upper)[0][-1] # last
                mask = np.logical_and(lower <= photon_index, photon_index <= upper)
    
                energies = energies[mask]
                peak_volts = peak_volts[mask]
                photon_index = photon_index[mask] - lower
                
            else:
                trace_slice = trace
                trace_time_slice = trace_time
                
            # Basic Plots ##################################################################################################
            if n == 0:
                if plot_true_volts:
                    label = 'True Peak Volts -- Vpp: {}, Stretch: {:.3f}'.format(vpp, factor)
                    axes[axis_count, col].plot(trace_time[photon_index], peak_volts, label=label)
                    axes[axis_count, col].legend()
                    
                if plot_trace:
                    label = 'Trace -- Vpp: {}, Stretch: {:.3f}'.format(vpp, factor)
                    axes[axis_count, col].plot(trace_time_slice, trace_slice, label=label)
                    axes[axis_count, col].legend()
                
            if plot_true_volts or plot_trace:
                axis_count += 1

            # Deconvolution ###############################################################################################
            deconv = td_nnlsr_deconvolve(trace_slice - baseline, kernel)
            # deconv += baseline
            
            if n == 0:
                if plot_deconv:
                    label = 'Deconv -- Vpp: {}, Stretch: {:.3f}'.format(vpp, factor)
                    axes[axis_count, col].plot(trace_time_slice, deconv, marker='.', linestyle='', label=label)

            # volts_trace = baseline * np.ones_like(deconv)
            # volts_trace[photon_index] = peak_volts
            volts_trace = np.zeros_like(deconv)
            volts_trace[photon_index] += peak_volts - baseline

            error = np.abs(volts_trace - deconv)
            error_mask = np.where(np.abs(error) > error_threshold)
            
            if n == 0:
                if plot_error:
                    axes[axis_count, col].plot(trace_time_slice[error_mask], error[error_mask],
                                               marker = '.', linestyle='', color='red', label='Error', alpha=.5)

            if plot_deconv or plot_error:
                axis_count += 1

            # Histograms ####################################################################################################   
            true_hist, bin_edges = np.histogram(peak_volts-baseline, bins=bins)
            true = true + true_hist
            deconv_hist, bin_edges = np.histogram(deconv, bins=bins)
            counted = counted + deconv_hist
            
        if plot_hists:
            axis = axes[-1, col]
            
            label = 'Ground Truth'
            my_histplot(axis, true, bin_edges, alpha=.8, label=label)

            label = 'Vpp: {}, Stretch: {:.3f}'.format(vpp, factor)
            my_histplot(axis, counted, bin_edges, alpha=.8, label=label)

            # axis.set_xlim(0, bin_edges[max(np.where(true >= 1)[0][-1], np.where(counted >= 1)[0][-1])])
            # axis.set_xlim(0, vpp * 1000)
            
            if vpp == 1:
                axis.set_xlim(0, 400)
            else:
                axis.set_xlim(0, 400)
            
            axis.set_yscale('log')
            axis.legend()
            axis.set_ylabel('Counts')
            axis.set_xlabel('mV from Baseline, {:.2f} mV bins'.format(bins[1] - bins[0]))
                


In [ ]:
# def test_vpp_and_compression(fwhm, spectrum,binenergies, dt, tstep,
#                              trace_length, mV_per_ADC, keV_per_area, specscale_keV,
#                              baseline, basenoise, bits,mean,std, vpp, kernel, countrates):
# 
#     for countrate in countrates:
# 
#         fig, axes = plt.subplots(3, 1, figsize=(8, 6), dpi=200)
# 
#         counts = int(countrate*TGF_duration) #total counts incident on the detector
# 
#         # Build Trace #############################################################################################################
#         # Build trace and find trigger time
#         trace_time, trace, photon_index, energies, peak_volts = lognormal_nai_trace(counts, fwhm, spectrum,binenergies, dt, tstep,
#                                                                              trace_length, vpp*mV_per_ADC, keV_per_area, specscale_keV,
#                                                                              baseline, basenoise, bits,mean,std, kernel, vpp=vpp)
# 
#         # print(photon_index, trace_time[photon_index], trace.size)
# 
#         trace_time *= 1E6 # convert to microseconds    
#         trigger_time, trigger_index = trace_trigger(trace, trace_time)
# 
#         # Get Trace slice with significant activity
#         lower = 100
#         upper = 140
#         mask = np.logical_and(lower <= trace_time, trace_time <= upper)
#         trace_slice = trace[mask]
#         trace_time_slice = trace_time[mask]
# 
#         lower = np.nonzero(lower <= trace_time)[0][0] #first
#         upper = np.nonzero(trace_time <= upper)[0][-1] # last
#         mask = np.logical_and(lower <= photon_index, photon_index <= upper)
# 
#         energies = energies[mask]
#         peak_volts = peak_volts[mask]
#         photon_index = photon_index[mask] - lower
# 
#         axis = axes[0]
#         axis.plot(trace_time_slice, trace_slice, color='black', marker='.', markersize=2, linestyle='',
#                   label='Countrate: {:.1e}, Vpp: {}'.format(countrate, vpp))
#         # axis.set_xlim(100,300)
#         axis.set_ylabel('mV')
#         axis.legend()
#         axis.set_title('Simulated NaI trace data',fontsize=20)
#         axis.set_xlabel('microseconds',fontsize=20)
# 
#         # Deconvolution ###############################################################################################
#         deconv = td_nnlsr_deconvolve(trace_slice - baseline, kernel)
#         # deconv += baseline
# 
#         # volts_trace = baseline * np.ones_like(deconv)
#         # volts_trace[photon_index] = peak_volts
#         volts_trace = np.zeros_like(deconv)
#         volts_trace[photon_index] += peak_volts - baseline
# 
#         error = np.abs(volts_trace - deconv)
#         error_mask = np.where(np.abs(error) > 1)
# 
#         axis = axes[1]
# 
#         axis.plot(trace_time_slice[photon_index], peak_volts-baseline, 'g.', label='Truth [Peak Volts]')
#         axis.plot(trace_time_slice.squeeze(), deconv.squeeze(), 'b.', label='Deconvolution', alpha=.5)
#         axis.plot([-1], [-1], 'r.', label='Local Error') # for legend...
#         axis.set_xlim(np.min(trace_time_slice), np.max(trace_time_slice))
#         axis.legend()
#         axis.set_ylabel('mV From Baseline')
# 
#         gc.collect()
# 
#         # Histograms ####################################################################################################
#         if bits == 8:
#             bins = np.linspace(1, vpp*1000, 256)
#         else:
#             bins = np.linspace(1, vpp*1000, 2**bits)
#             
#         print(np.max(bins), bins.size)
# 
#         axis = axes[2]
# 
#         hist, bin_edges = np.histogram(peak_volts-baseline, bins=bins)
#         # make duplicate points while using normal plot function for histogram
#         x = np.stack((bin_edges[1:], bin_edges[:-1])).transpose()[:,::-1].ravel() 
#         y = np.stack((hist, hist)).transpose().ravel()
#         x = np.insert(x, 0, 0)
#         y = np.insert(y, 0, 0)
#         axis.plot(x, y, alpha=.8, label='Truth [Peak Volts] Hist')
# 
#         hist, bin_edges = np.histogram(deconv, bins=bins)
#         x = np.stack((bin_edges[1:], bin_edges[:-1])).transpose()[:,::-1].ravel()
#         y = np.stack((hist, hist)).transpose().ravel()
#         x = np.insert(x, 0, 0)
#         y = np.insert(y, 0, 0)
#         axis.plot(x, y, alpha=.8, label='Deconvolution Hist')
# 
#         axis.set_yscale('log')
#         axis.legend()
#         axis.set_ylabel('Counts')
#         axis.set_xlabel('mV from Baseline, {:.2f} mV wide bins'.format(bins[1] - bins[0]))
#         # assert False
# 
#         gc.collect()

In [ ]:
# countrates = np.logspace(6,8,5)

In [ ]:
# test_vpp_and_compression(fwhm, spectrum,binenergies, dt, tstep,
#                              trace_length, mV_per_ADC, keV_per_area, specscale_keV,
#                              baseline, basenoise, bits,mean,std, vpp=1, kernel=pulse, countrates=countrates)

In [ ]:
# test_vpp_and_compression(fwhm, spectrum,binenergies, dt, tstep,
#                              trace_length, mV_per_ADC, keV_per_area, specscale_keV,
#                              baseline, basenoise, bits,mean,std, vpp=4, kernel=pulse, countrates=countrates)

In [ ]:
# test_vpp_and_compression(fwhm, spectrum,binenergies, dt, tstep,
#                              trace_length, mV_per_ADC, keV_per_area, specscale_keV,
#                              baseline, basenoise, bits,mean,std, vpp=1, kernel=compressed_pulse, countrates=countrates)

In [ ]:
# test_vpp_and_compression(fwhm, spectrum,binenergies, dt, tstep,
#                              trace_length, mV_per_ADC, keV_per_area, specscale_keV,
#                              baseline, basenoise, bits,mean,std, vpp=1, kernel=stretched_pulse, countrates=countrates)

In [ ]:
# test_vpp_and_compression(fwhm, spectrum,binenergies, dt, tstep,
#                              trace_length, mV_per_ADC, keV_per_area, specscale_keV,
#                              baseline, basenoise, bits,mean,std, vpp=4, kernel=compressed_pulse, countrates=countrates)